# 🕸️ DeepFetch

**Universal web scraping & automation engine — self-hosted, API-first, stealth-ready.**

Run the two cells below. A public dashboard link will appear at the end.

> ⏱️ First run ≈ 5 min (installs Node 22 + Playwright). Subsequent runs: ~90 sec.


In [ ]:
# ═══════════════════════════════════════════════════════
#  Cell 1 — Install, clone & build
# ═══════════════════════════════════════════════════════
import subprocess, os, sys

def run(cmd, cwd=None):
    """Run a shell command, stream output, raise on failure."""
    proc = subprocess.run(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    if proc.returncode != 0:
        # Show last 80 lines to help debug
        lines = proc.stdout.splitlines()
        print("\n".join(lines[-80:]))
        raise RuntimeError(f"Failed: {cmd}")
    return proc.stdout.strip()

REPO   = "/deepfetch"
PWHOME = "/ms-playwright"
os.environ["PLAYWRIGHT_BROWSERS_PATH"] = PWHOME

# ── Node.js 22 ──────────────────────────────────────────
if not os.path.exists("/usr/bin/node") or "v22" not in run("node --version"):
    print("📦 Installing Node.js 22 ...")
    run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - 2>&1 | tail -3")
    run("apt-get install -y nodejs 2>&1 | tail -3")
print(f"✅ Node {run('node --version')}  npm {run('npm --version')}")

# ── cloudflared (zero-auth public tunnel) ───────────────
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("📦 Installing cloudflared ...")
    run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && "
        "chmod +x /usr/local/bin/cloudflared")
print(f"✅ cloudflared {run('cloudflared --version').split()[2]}")

# ── System libs for Playwright Chromium ─────────────────
print("📦 Installing system libs ...")
run("apt-get install -y --no-install-recommends "
    "libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 "
    "libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 "
    "libgbm1 libasound2 libpango-1.0-0 libpangocairo-1.0-0 "
    "2>&1 | tail -3")
print("✅ System libs ready")

# ── Clone / pull ─────────────────────────────────────────
if not os.path.exists(REPO):
    print("📥 Cloning ferelking242/deepfetch ...")
    run(f"git clone --depth 1 https://github.com/ferelking242/deepfetch {REPO}")
else:
    print("📥 Pulling latest ...")
    run("git pull --ff-only", cwd=REPO)

# ── npm install ──────────────────────────────────────────
print("📦 npm install (server) ...")
run("npm install 2>&1 | tail -5", cwd=f"{REPO}/server")

print("📦 npm install (dashboard) ...")
run("npm install 2>&1 | tail -5", cwd=f"{REPO}/dashboard")

# ── Playwright Chromium ──────────────────────────────────
if not any(os.path.exists(f"{PWHOME}/chromium-{v}")
           for v in os.listdir(PWHOME) if v.startswith("chromium"))
           if os.path.exists(PWHOME) else True:
    print("🎭 Installing Playwright Chromium ...")
    run("npx playwright install chromium --with-deps 2>&1 | tail -8",
        cwd=f"{REPO}/server")
print("✅ Playwright ready")

# ── Build TypeScript (use local tsc binary directly) ─────
print("🔨 Building server ...")
run("./node_modules/.bin/tsc --project tsconfig.json", cwd=f"{REPO}/server")
print("✅ Server built")

print("🔨 Building dashboard ...")
run("npm run build 2>&1 | tail -5", cwd=f"{REPO}/dashboard")
print("✅ Dashboard built")

print()
print("🎉 Build complete — run Cell 2 to launch.")


In [ ]:
# ═══════════════════════════════════════════════════════
#  Cell 2 — Launch DeepFetch + public tunnel
#  Your dashboard link appears at the bottom.
# ═══════════════════════════════════════════════════════
import subprocess, os, time, re, secrets, yaml

try:
    import yaml
except ImportError:
    subprocess.run("pip install pyyaml -q", shell=True)
    import yaml

REPO    = "/deepfetch"
PORT    = 3000
DATA    = "/deepfetch-data"
CFG     = f"{REPO}/config.yaml"
os.makedirs(DATA, exist_ok=True)
os.environ["PLAYWRIGHT_BROWSERS_PATH"] = "/ms-playwright"

# ── Kill any leftovers ───────────────────────────────────
subprocess.run("pkill -f 'node.*dist/index' 2>/dev/null; "
               "pkill -f cloudflared 2>/dev/null",
               shell=True, capture_output=True)
time.sleep(1)

# ── Write config only if missing or regenerate ───────────
if not os.path.exists(CFG):
    print("⚙️  Generating config.yaml ...")
    cfg = {
        "server":   {"port": PORT, "host": "0.0.0.0",
                     "master_secret": secrets.token_hex(32)},
        "browser":  {"pool_max": 0, "pool_reserved": 1,
                     "context_ttl_seconds": 300,
                     "navigation_timeout_ms": 30000, "headless": True},
        "resources":{"cpu_threshold_pct": 85, "ram_threshold_pct": 80},
        "queue":    {"max_retries": 3, "retry_base_delay_ms": 2000,
                     "result_ttl_seconds": 86400},
        "ai_engine":{"enabled": True, "trigger": "on_selector_failure",
                     "max_html_chars": 50000, "timeout_ms": 15000,
                     "providers": [
                         {"name": "ollama", "local": True,
                          "model": "llama3.2", "base_url": "http://localhost:11434"},
                         {"name": "groq", "api_key": "",
                          "model": "llama-3.3-70b-versatile"},
                     ]},
        "zeusdl":   {"binary": "zeusdl", "extra_flags": []},
        "sessions": {"encryption_key": secrets.token_hex(32),
                     "check_interval_seconds": 1800},
        "data_dir": DATA,
    }
    with open(CFG, "w") as f:
        yaml.dump(cfg, f)
    print("   Config written. Edit AI keys from the dashboard.")
else:
    print("⚙️  Using existing config.yaml")

# ── Start server ─────────────────────────────────────────
print("🚀 Starting DeepFetch ...")
log = open("/tmp/deepfetch.log", "w")
srv = subprocess.Popen(
    ["node", f"{REPO}/server/dist/index.js"],
    env={**os.environ, "DF_CONFIG": CFG, "NODE_ENV": "production"},
    stdout=log, stderr=log, cwd=REPO,
)

# Wait up to 30s for the server to be ready
import urllib.request, urllib.error
for i in range(30):
    time.sleep(1)
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/v1/health", timeout=2)
        print(f"   ✅ Server up ({i+1}s)")
        break
    except:
        pass
else:
    log.flush()
    print("❌ Server failed. Last logs:")
    print(open("/tmp/deepfetch.log").read()[-3000:])
    raise RuntimeError("DeepFetch did not start")

# ── cloudflared tunnel ───────────────────────────────────
print("🌐 Opening public tunnel ...")
cf_log = open("/tmp/cf.log", "w")
subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=cf_log, stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
for _ in range(30):
    time.sleep(1)
    try:
        content = open("/tmp/cf.log").read()
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', content)
        if m:
            PUBLIC_URL = m.group(0)
            break
    except:
        pass

if not PUBLIC_URL:
    print("⚠️  cloudflared URL not found — check /tmp/cf.log")
    PUBLIC_URL = f"http://localhost:{PORT}  (local only)"

# ── Done ─────────────────────────────────────────────────
print()
print("═" * 52)
print("  🎉  DeepFetch is running")
print("═" * 52)
print(f"  📊  Dashboard : {PUBLIC_URL}/dashboard")
print(f"  📖  API Docs  : {PUBLIC_URL}/docs")
print(f"  ❤️   Health    : {PUBLIC_URL}/v1/health")
print("═" * 52)
print()
print("Open the Dashboard link on any device (phone, tablet…)")
print("Configure AI keys, sessions and everything else there.")
